# Download the TCV confinement state dataset

Run this once. It downloads `data.zip` from Zenodo into memory, opens each `TCV_confstate_<shot>.parquet` inside, keeps only the shots in `SHOTS_KEEP` and the columns in `COLS_KEEP`, and writes the filtered parquets to `DATA_DIR/parquets/`.

In [ ]:
COLS_KEEP = ["time", "Halpha13", "IPLA", "FIR_LIDs_core", "INPWR", "BETAT", "label_conf"]
SHOTS_KEEP = [29511, 30225, 30262, 31839, 43454, 48580, 57013, 57077, 57093, 57732, 58460, 59065, 59076, 59825, 60097, 60813, 60995, 61000, 61021, 61039, 61246, 61260, 61279, 63246, 63308, 63923, 63944, 64370, 64372, 64386, 64820, 65481, 65564, 66494, 68643, 69514, 69515, 73330, 73631, 73931, 76702, 77602, 78069, 79826, 79827]

In [ ]:
import io, re, urllib.request, zipfile
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = Path.cwd()  # this notebook lives in TCV/data/
OUT_DIR = DATA_DIR / "parquets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://zenodo.org/records/16631053/files/data.zip?download=1"
FILENAME_RE = re.compile(r"TCV_confstate_(\d+)\.parquet$")
shots_keep = set(SHOTS_KEEP)

print(f"Downloading {DATASET_URL} ...")
buf = io.BytesIO()
with urllib.request.urlopen(DATASET_URL) as resp:
    total = int(resp.headers.get("Content-Length") or 0) or None
    with tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024,
              desc="download") as pbar:
        for chunk in iter(lambda: resp.read(1 << 16), b""):
            buf.write(chunk)
            pbar.update(len(chunk))
zip_bytes = buf.getvalue()
print(f"Downloaded {len(zip_bytes) / 1e6:.1f} MB; processing in memory ...")

n_written = 0
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    wanted = [m for m in z.namelist()
              if (mm := FILENAME_RE.search(m)) and int(mm.group(1)) in shots_keep]
    for member in tqdm(wanted, desc="extract"):
        shot = int(FILENAME_RE.search(member).group(1))
        with z.open(member) as f:
            df = pd.read_parquet(io.BytesIO(f.read()))
        df = df[[c for c in COLS_KEEP if c in df.columns]]
        df.to_parquet(OUT_DIR / f"TCV_confstate_{shot}.parquet")
        n_written += 1

missing = sorted(shots_keep - {int(FILENAME_RE.search(p.name).group(1))
                               for p in OUT_DIR.glob("TCV_confstate_*.parquet")})
print(f"Wrote {n_written} parquets to {OUT_DIR}")
if missing:
    print(f"Warning: {len(missing)} requested shots not found in zip: {missing}")